# 05.06 Active 4-Model Comparison (LR, CatBoost, EBM, Soft-Voting Ensemble)

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
TRAINED_DIR = PROJECT_ROOT / "models" / "trained"
REPORTS_DIR = PROJECT_ROOT / "reports" / "phase_05"
VIZ_DIR = REPORTS_DIR / "visualizations" / "model_comparison"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

artifacts = [
    ("Logistic Regression", MODELS_DIR / "lr_metrics.json"),
    ("CatBoost", TRAINED_DIR / "gradient_boosting" / "catboost_metrics.json"),
    ("Explainable Boosting Machine (EBM)", TRAINED_DIR / "interpretable" / "ebm_metrics.json"),
    ("Soft-Voting Ensemble", MODELS_DIR / "ensemble_metrics.json"),
]

rows = []
for name, p in artifacts:
    d = json.loads(p.read_text(encoding="utf-8"))
    ta = d["track_a_oof"]
    tb = d["track_b_oof"]
    fstd = d["track_a_fold_std"]
    rows.append({
        "model": name,
        "trackA_accuracy": round(ta["accuracy"], 4),
        "trackA_precision": round(ta["precision"], 4),
        "trackA_recall": round(ta["recall"], 4),
        "trackA_specificity": round(ta["specificity"], 4),
        "trackA_balanced_acc": round(ta["balanced_accuracy"], 4),
        "trackA_f1": round(ta["f1"], 4),
        "trackA_macro_f1": round(ta["macro_f1"], 4),
        "trackA_macro_f1_fold_std": round(fstd["macro_f1"], 4),
        "trackA_roc_auc": round(ta["roc_auc"], 4),
        "trackA_brier": round(ta["brier_score"], 4),
        "trackA_log_loss": round(ta["log_loss"], 4),
        "trackA_ece": round(ta["ece"], 4),
        "trackB_macro_f1": round(tb["macro_f1"], 4),
        "trackB_roc_auc": round(tb["roc_auc"], 4),
        "trackB_brier": round(tb["brier_score"], 4),
    })

df_comp = pd.DataFrame(rows).sort_values(["trackA_macro_f1", "trackA_roc_auc"], ascending=False).reset_index(drop=True)
df_comp["rank"] = np.arange(1, len(df_comp) + 1)
df_comp.to_csv(REPORTS_DIR / "model_evaluation_comparison.csv", index=False)
df_comp.to_csv(VIZ_DIR / "model_comparison_summary_table.csv", index=False)

champion = df_comp.iloc[0].to_dict()
summary = {
    "evaluation_scope": "4_active_models_authentic_patient_grouped_10fold_cv",
    "champion_selection": {
        "champion_model": champion["model"],
        "champion_macro_f1": champion["trackA_macro_f1"],
        "champion_roc_auc": champion["trackA_roc_auc"],
        "champion_brier": champion["trackA_brier"],
    },
    "model_rankings": df_comp.to_dict(orient="records"),
}
(TRAINED_DIR / "model_comparison_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(df_comp[["rank", "model", "trackA_accuracy", "trackA_macro_f1", "trackA_roc_auc", "trackA_brier", "trackB_macro_f1"]].to_string(index=False))
print(f"\nChampion Model (Track A 10-Fold OOF): {champion['model']}")

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(df_comp))
w = 0.35
ax.bar(x - w/2, df_comp["trackA_macro_f1"], w, label="Track A (Authentic-Only)", color="#2b5c8f", edgecolor="black")
ax.bar(x + w/2, df_comp["trackB_macro_f1"], w, label="Track B (Synthetic-Augmented Train)", color="#41b6c4", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(df_comp["model"], rotation=15, ha="right")
ax.set_ylim(0, 1.1); ax.set_ylabel("10-Fold OOF Macro F1")
ax.set_title("Active 4-Model Comparison: Track A vs Track B (Authentic-Only Validation)")
ax.legend()
plt.tight_layout(); plt.savefig(VIZ_DIR / "graph1_macro_f1_comparison.png", dpi=200); plt.close(fig)


 rank                              model  trackA_accuracy  trackA_macro_f1  trackA_roc_auc  trackA_brier  trackB_macro_f1
    1                           CatBoost           0.9474           0.9424          0.9600        0.0514           0.9313
    2               Soft-Voting Ensemble           0.9263           0.9193          0.9490        0.0579           0.9084
    3 Explainable Boosting Machine (EBM)           0.9263           0.9193          0.9214        0.0709           0.9084
    4                Logistic Regression           0.9158           0.9105          0.9490        0.0768           0.8763

Champion Model (Track A 10-Fold OOF): CatBoost
